# Chạy Tự Động 1 Lượt Trên Colab Web (`Runtime -> Run all`)

Không cần chỉnh sửa code — chỉ cần chọn **Runtime → Change runtime type → L4 GPU**, sau đó bấm **Runtime → Run all (`Ctrl + F9`)**:
- **Cell 1:** Tải code mới nhất từ GitHub + kết nối Google Drive (lưu vĩnh viễn LoRA & video) + cài thư viện.
- **Cell 2:** Train LoRA từ 12 ảnh nhân vật *(tự bỏ qua nếu trên Google Drive đã có sẵn file LoRA)*.
- **Cell 3:** Sinh 2 ảnh Keyframe hành động mới từ LoRA.
- **Cell 4:** Tạo giọng đọc biểu cảm theo hoàn cảnh.
- **Cell 5:** Tạo chuyển động video AI (`Wan 2.2` trên L4 hoặc `SVD-XT` trên T4).
- **Cell 6:** Ghép video hoàn chỉnh, lưu về Google Drive, mở link xem & tự tải file về máy tính.

In [ ]:
# CELL 1: KÉO CODE GITHUB + KẾT NỐI GOOGLE DRIVE + CÀI THƯ VIỆN
import os
REPO_URL = 'https://github.com/nguyenkhanhan123/Test.git'
REPO_DIR = '/content/Test'
PROJECT_DIR = '/content/Test/ai_film_pipeline'

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('Bỏ qua mount Drive:', e)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {PROJECT_DIR}
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q --upgrade edge-tts pysubs2 pydantic imageio-ffmpeg mutagen gradio peft bitsandbytes accelerate transformers sentencepiece ftfy git+https://github.com/huggingface/diffusers.git
!python prepare_user_assets.py --trigger nv_minhkhang

In [ ]:
# CELL 2: TRAIN CHARACTER LORA (TỰ BỎ QUA NẾU ĐÃ CÓ TRÊN GOOGLE DRIVE)
%cd /content/Test/ai_film_pipeline
import os, shutil

local_lora = 'loras/nv_minhkhang_lora_v1/pytorch_lora_weights.safetensors'
drive_lora = '/content/drive/MyDrive/ai_film_loras/nv_minhkhang_lora_v1.safetensors'
FORCE_RETRAIN = False

if not os.path.exists(local_lora) and os.path.exists(drive_lora):
    os.makedirs(os.path.dirname(local_lora), exist_ok=True)
    shutil.copy2(drive_lora, local_lora)
    print('[LoRA] Đã lấy lại LoRA từ Google Drive:', local_lora)

if os.path.exists(local_lora) and not FORCE_RETRAIN:
    print(f'[LoRA] Đã có sẵn {local_lora} ({os.path.getsize(local_lora)/1e6:.2f} MB) -> Bỏ qua bước train!')
else:
    !python cloud_scripts/train_sdxl_lora_peft.py \
      --dataset user_inputs/photos \
      --output loras/nv_minhkhang_lora_v1 \
      --steps 360 \
      --resolution 768
    if os.path.exists(local_lora) and os.path.exists('/content/drive/MyDrive'):
        os.makedirs('/content/drive/MyDrive/ai_film_loras', exist_ok=True)
        shutil.copy2(local_lora, drive_lora)
        print('[DRIVE SAVED] Đã cất vĩnh viễn LoRA sang Google Drive!')

In [ ]:
# CELL 3: SINH 2 ẢNH KEYFRAME HÀNH ĐỘNG MỚI TỪ LORA
%cd /content/Test/ai_film_pipeline
!pip uninstall -y -q torchao
!python cloud_scripts/step4_gen_keyframes_lora.py

In [ ]:
# CELL 4: TẠO GIỌNG ĐỌC BIỂU CẢM & KHÓA THỜI LƯỢNG TỪNG CẢNH
%cd /content/Test/ai_film_pipeline
from pathlib import Path
voices_ready = list(Path('user_inputs/voices').glob('*.wav'))
if voices_ready:
    !pip install -q git+https://github.com/danhtran2mind/F5-TTS.git whisperx
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='danhtran2mind/Vi-F5-TTS', local_dir='/content/Vi-F5-TTS')
    from cloud_scripts.step3_cloud_vi_f5_tts import clone_episode_voices_vi_f5
    clone_episode_voices_vi_f5(Path('.'), Path('script_action_demo.json'), Path('/content/Vi-F5-TTS'))
else:
    !python step_action_demo.py --tts-only

In [ ]:
# CELL 5: TẠO VIDEO HÀNH ĐỘNG AI (WAN 2.2 TRÊN L4 HOẶC SVD-XT TRÊN T4)
%cd /content/Test/ai_film_pipeline
!pip install -q ftfy
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python cloud_scripts/step5_batch_wan22_diffusers.py --script script_action_demo.json --steps 24

In [ ]:
# CELL 6: GHÉP VIDEO HOÀN CHỈNH, LƯU DRIVE, MỞ LINK XEM & TỰ TẢI VỀ MÁY
%cd /content/Test/ai_film_pipeline
import json, os, re, shutil, subprocess, threading, time
from IPython.display import Image as IPyImage, display

subprocess.run(['python', 'step_action_demo.py', '--skip-tts'], check=True)
sc = json.loads(open('script_action_demo.json', encoding='utf-8').read())
print('=== TRẠNG THÁI TỪNG CẢNH ===')
for s in sc['shots']:
    print(f"  {s['shot_id']}: {s.get('render_mode_used')} ({s.get('duration_sec')}s)")

for kf in ['assets/keyframes_lora/shot_103.png', 'assets/keyframes_lora/shot_104.png']:
    if os.path.exists(kf):
        display(IPyImage(filename=kf, width=480))

video_out = 'output/action_pilot_demo.mp4'
lora_out = 'loras/nv_minhkhang_lora_v1/pytorch_lora_weights.safetensors'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/ai_film_loras', exist_ok=True)
    shutil.copy2(video_out, '/content/drive/MyDrive/ai_film_loras/action_pilot_demo.mp4')
    if os.path.exists(lora_out):
        shutil.copy2(lora_out, '/content/drive/MyDrive/ai_film_loras/nv_minhkhang_lora_v1.safetensors')
    print('[DRIVE SAVED] MyDrive/ai_film_loras/action_pilot_demo.mp4')

try:
    from google.colab import files
    files.download(video_out)
except Exception as e:
    print('Không tự download qua browser được:', e)

import gradio as gr
with gr.Blocks(title='AI Action Video Demo') as demo:
    gr.Markdown('## Video Hành Động Hoàn Chỉnh + Ảnh LoRA Đã Train')
    gr.Video(value=video_out, label='Video Hoàn Chỉnh (action_pilot_demo.mp4)')
    dl_files = [p for p in [video_out, lora_out] if os.path.exists(p)]
    gr.File(value=dl_files, label='Tải Video (.mp4) & LoRA (.safetensors)')

demo.launch(server_name='127.0.0.1', share=False, prevent_thread_lock=True)
port = demo.server_port
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
cf_proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
t0 = time.time()
while time.time() - t0 < 25:
    line = cf_proc.stdout.readline()
    if not line:
        break
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in cf_proc.stdout], daemon=True).start()
print('====================================================')
print('LINK XEM & TẢI VIDEO:', public_url or f'http://127.0.0.1:{port}')
print('====================================================')